# NB03 — explicit graph projection
Requires this checkout and the `lmdb` extra. We store an untyped Alice and a worksFor edge,
map only that relationship to a logical predicate, then infer Researcher and Person membership.
Expected boundary: two memberships in the logical closure, still no stored Person label.
The final figure is a caller-owned view of facts, not a database mutation or RDF export.

In [ ]:
from tempfile import TemporaryDirectory
from gestaltdb.graphdb import Edge, GraphDB, Node
from gestaltdb.ontology import Domain, Ontology, SubClassOf, TYPE_PREDICATE, compile_ontology
from gestaltdb.reasoning import Fact, ReasoningLimits
from gestaltdb.reasoning.adapters.graph import GraphMapping, PinnedGraphSource
from gestaltdb.viz.api import VizOptions, visualize_nodes_edges

plan = compile_ontology(Ontology('people', [Domain('worksFor', 'Researcher'), SubClassOf('Researcher', 'Person')]))
with TemporaryDirectory() as directory:
    graph = GraphDB.create(directory, backend='lmdb', serializer='json')
    try:
        graph.put_nodes([Node('alice'), Node('acme')])
        graph.put_edge(Edge('e1', 'alice', 'acme', {'type': 'worksFor'}))
        with graph.current_read_view(require_verifiable=True) as view:
            with PinnedGraphSource(view, mapping=GraphMapping(edge_types=[('worksFor', 'worksFor')])) as source:
                scoped = source.evaluate(plan, limits=ReasoningLimits(), collect_proofs=True)
        assert scoped.match(subject='alice', predicate=TYPE_PREDICATE) == tuple(sorted([
            Fact('alice', TYPE_PREDICATE, 'Person'), Fact('alice', TYPE_PREDICATE, 'Researcher')]))
        assert graph.get_node(b'alice').labels == ()
        assert not graph.query('MATCH (a:Person) RETURN a').records
    finally:
        graph.close()

facts = scoped.result.facts
nodes = [Node(node_id=symbol, properties={'name': symbol})
         for symbol in sorted({f.subject for f in facts} | {f.object for f in facts})]
edges = [Edge(edge_id=f'logical-{i}', source=f.subject, target=f.object,
              properties={'type': 'rdf:type' if f.predicate == TYPE_PREDICATE else f.predicate})
         for i, f in enumerate(facts)]
figure = visualize_nodes_edges(nodes, edges, options=VizOptions(node_label_property='name', edge_labels='all'))
svg = figure.as_svg()
assert '<svg' in svg and 'Researcher' in svg
figure


The read view stays open through extraction/evaluation; the immutable result survives its closure.
The notebook displays self-contained HTML via GestaltDB's VizFigure.
`svg` is a static representation; it does not capture changes made in the browser widget.